# 📊 Module 1: Data Collection & Data Cleaning
### Customer Churn Prediction & Business Analytics
**Portfolio Track**: Enterprise Data Science Portfolio  
**Author**: Data Science Apprentice  
**Dataset**: IBM Telco Customer Churn (7,043 customer records, 21 initial attributes)

---

## 🎯 Objectives
1. **Load Raw Data**: Ingest the telecom dataset (`WA_Fn-UseC_-Telco-Customer-Churn.csv`).
2. **Structural Inspection**: Verify dimensions, data types, column distributions, and null values.
3. **Data Quality Audit**:
   - Detect whitespace/blank characters disguised as valid entries.
   - Investigate anomalies (e.g., `TotalCharges` stored as `object`).
   - Check for duplicate customer records.
4. **Data Cleansing & Type Casts**:
   - Convert `TotalCharges` to `float64` and handle edge cases (new customers with `tenure == 0`).
   - Standardize categorical string representations.
5. **Feature Formatting & Export**: Export cleaned dataset (`data/cleaned_telco_churn.csv`) for downstream EDA, statistical analysis, and machine learning.


In [1]:
import os
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

print("Environment Ready:")
print(f"  Pandas version: {pd.__version__}")
print(f"  NumPy version:  {np.__version__}")


Environment Ready:
  Pandas version: 2.1.4
  NumPy version:  1.26.4


## 1. Data Ingestion
We load the raw CSV dataset from `data/WA_Fn-UseC_-Telco-Customer-Churn.csv`.

In [2]:
data_path = 'data/WA_Fn-UseC_-Telco-Customer-Churn.csv'
if not os.path.exists(data_path):
    data_path = '../data/WA_Fn-UseC_-Telco-Customer-Churn.csv'

df_raw = pd.read_csv(data_path)
print(f"Loaded raw dataset with {df_raw.shape[0]} rows and {df_raw.shape[1]} columns.")
df_raw.head()


Loaded raw dataset with 7043 rows and 21 columns.


## 2. Dataset Overview & Data Quality Audit
Let's check the schema, data types, non-null counts, and uniqueness.

In [3]:
print("Dataset Info:")
df_raw.info()


Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-n

In [4]:
print("Summary Statistics for Numeric Attributes:")
df_raw.describe()


Summary Statistics for Numeric Attributes:


In [5]:
print(f"Total Rows: {len(df_raw)}")
print(f"Unique Customer IDs: {df_raw['customerID'].nunique()}")
print(f"Duplicate Rows Count: {df_raw.duplicated().sum()}")


Total Rows: 7043
Unique Customer IDs: 7043
Duplicate Rows Count: 0


## 3. Identification & Resolution of TotalCharges Anomaly
Notice in `.info()` that `TotalCharges` is an `object` (string) rather than `float64`.
Let us investigate why it was not parsed as a floating-point number.

In [6]:
whitespace_mask = df_raw['TotalCharges'].str.strip() == ''
print(f"Number of rows with empty whitespace string in TotalCharges: {whitespace_mask.sum()}")

# Inspect these anomalous records
anomalous_records = df_raw[whitespace_mask][['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges', 'Contract', 'Churn']]
print("Anomalous Records:")
print(anomalous_records)


Number of rows with empty whitespace string in TotalCharges: 11
Anomalous Records:
      customerID  tenure  MonthlyCharges TotalCharges  Contract Churn
488   4472-LVYGI       0           52.55               Two year    No
753   3115-CZMZD       0           20.25               Two year    No
936   5709-LVOEQ       0           80.85               Two year    No
1082  4367-NUYAO       0           25.75               Two year    No
1340  1371-DWPAZ       0           56.05               Two year    No
3331  7644-OMVMY       0           19.85               Two year    No
3826  3213-VVOLG       0           25.35               Two year    No
4380  2520-SGTTA       0           20.00               Two year    No
5218  2923-ARZLG       0           19.70               One year    No
6670  4075-WKNIU       0           73.35               Two year    No
6754  2775-SEFEE       0           61.90               Two year    No


### 💡 Business Insight on Missing TotalCharges:
All 11 customers with missing `TotalCharges` have `tenure = 0` months. These are newly registered accounts who have not yet reached their first billing cycle.
Therefore, setting their `TotalCharges` to `0.0` is mathematically and domain-wise accurate, avoiding deletion of recent signups.

## 4. Execution of Cleaning Steps
1. Convert `TotalCharges` to numeric, setting whitespace to NaN.
2. Impute `0.0` for new customers with `tenure == 0`.
3. Strip whitespace from all string features.
4. Verify null counts and confirm data types.
5. Create binary target column `Churn_Binary` (1 = Yes, 0 = No) for numerical evaluation.
6. Create structured `TenureGroup` cohorts for business segmentation.

In [7]:
df_clean = df_raw.copy()

# 1. Convert TotalCharges to float64
df_clean['TotalCharges'] = pd.to_numeric(df_clean['TotalCharges'], errors='coerce')

# 2. Impute 0.0 for tenure == 0
df_clean.loc[(df_clean['tenure'] == 0) & (df_clean['TotalCharges'].isnull()), 'TotalCharges'] = 0.0

# 3. Strip whitespace across object columns
str_cols = df_clean.select_dtypes(include='object').columns
for col in str_cols:
    df_clean[col] = df_clean[col].str.strip()

# 4. Binary encoding for target
df_clean['Churn_Binary'] = df_clean['Churn'].map({'Yes': 1, 'No': 0}).astype(int)

# 5. Tenure Cohort Binning
df_clean['TenureGroup'] = pd.cut(
    df_clean['tenure'],
    bins=[-1, 12, 24, 36, 48, 60, 72],
    labels=['0-12 Mo', '13-24 Mo', '25-36 Mo', '37-48 Mo', '49-60 Mo', '61-72 Mo']
)

print("Data Cleaning Completed Successfully!")
print("Remaining Null Values per Column:")
null_summary = df_clean.isnull().sum()
print(null_summary[null_summary > 0] if (null_summary > 0).any() else "No missing values found!")


Data Cleaning Completed Successfully!
Remaining Null Values per Column:
No missing values found!


## 5. Verification & Clean Data Export
We verify statistical sanity on the cleaned dataset and export it to `data/cleaned_telco_churn.csv`.

In [8]:
print("Cleaned Dataset Numeric Summary:")
print(df_clean[['tenure', 'MonthlyCharges', 'TotalCharges', 'Churn_Binary']].describe())


Cleaned Dataset Numeric Summary:
       tenure  MonthlyCharges  TotalCharges  Churn_Binary
count 7043.00         7043.00       7043.00       7043.00
mean    32.37           64.76       2279.73          0.27
std     24.56           30.09       2266.79          0.44
min      0.00           18.25          0.00          0.00
25%      9.00           35.50        398.55          0.00
50%     29.00           70.35       1394.55          0.00
75%     55.00           89.85       3786.60          1.00
max     72.00          118.75       8684.80          1.00


In [9]:
out_path = 'data/cleaned_telco_churn.csv'
if not os.path.exists('data'):
    os.makedirs('data')

df_clean.to_csv(out_path, index=False)
print(f"Cleaned dataset saved to: {out_path} ({len(df_clean)} rows, {df_clean.shape[1]} columns)")


Cleaned dataset saved to: data/cleaned_telco_churn.csv (7043 rows, 23 columns)


### 📋 Summary of Module 1 Deliverables:
- **Raw Records Processed**: 7,043 customers, 21 columns.
- **Data Quality Issues Fixed**: `TotalCharges` resolved from text to float with 11 tenure=0 records imputed to 0.0.
- **Duplicates**: Verified 0 duplicates.
- **Engineered Columns**: `Churn_Binary` and `TenureGroup`.
- **Cleaned Data Shape**: (7043, 23).
- **Export**: Saved to `data/cleaned_telco_churn.csv`.
